# EMA + RSI

## Table of contents

- [Configuration](#configuration)
  - [Setup](#setup)
  - [Automatic](#automatic)
  - [Manual](#manual)
  - [Final configuration](#final-configuration)
- [EMA + RSI](#ema-rsi-section)
  - [Backtesting](#backtesting)
  - [Grid search](#grid-search)
  - [Walk-forward analysis](#walk-forward-analysis)
  - [Monte Carlo simulations](#monte-carlo-simulations)
  - [Live signals](#live-signals)
- [Inverse EMA + RSI](#inverse-ema--rsi)
  - [Backtesting](#inv-backtesting)
  - [Grid search](#inv-grid-search)
  - [Walk-forward analysis](#inv-walk-forward)
  - [Monte Carlo simulations](#inv-monte-carlo)
  - [Live signals](#inv-live-signals)

EMA Crossover + RSI Filter \
A momentum strategy that trades in the direction of the cross: fast EMA(9) over slow EMA(21) \
It uses ATR(14) for a dynamic volatility-adaptive trailing stop. \
Filters false signals with RSI(14) to cut whipsaws in ranging markets. \
Works across timeframes; well suited to scalping and intraday.

__How EMA+RSI Algorithm Determines Entry/Exit:__
- Fast EMA (9) / Slow EMA (21) – standard for crypto.
- Long Entry: Fast EMA crosses above Slow EMA AND RSI(14) < 70 (not overbought).
- Short Entry: Fast EMA crosses below Slow EMA AND RSI(14) > 30 (not oversold).
- Exit: Reverse crossover (signal flip) OR price hits the ATR-based trailing stop.
- The RSI filter reduces whipsaws in ranging markets.

__The RSI filter__ on ema / ema_inv is an optional, configurable filter:
- filter off entirely: EmaParams(rsi_filter=False)
- custom bounds: EmaParams(rsi_bullish=65.0, rsi_bearish=35.0)

## Configuration

### Setup

In [1]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
import dataclasses

from engine.backtester import Backtester
from engine.data_configurator import ACTIVE, load_data, save_result, LIVE_DIR
from engine.strategy_configurator import params_for, EXIT_PRESETS
from engine.trade_configurator import ACTIVE_TRADE
from engine.visualization import build_chart
from engine.evaluation import walk_forward, monte_carlo, grid_search
from engine.live import LiveEngine

import pandas as pd
import plotly.express as px

### Automatic

In [ ]:
# Automatic config: project-wide defaults defined by the three configurators.
# Override any of them in the manual chapter below; leave its dicts empty to stay automatic.
DATA_CONFIG     = ACTIVE              # engine/data_configurator.py     (DataSpec)
STRATEGY_CONFIG = params_for("ema")   # engine/strategy_configurator.py (EmaParams — this notebook's family)
TRADING_CONFIG  = ACTIVE_TRADE        # engine/trade_configurator.py    (TradingConfig)
EXIT_POLICY     = None                # None -> each strategy's assigned default (exit_policy_for)

### Manual


*_CONFIG = Automatic defaults, with any Manual overrides layered on top:
- Leave *_OVERRIDES empty → *_CONFIG is pure Automatic.
- Fill it → Automatic baseline + your Manual overrides.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — STRATEGY signal knobs. Empty = automatic EmaParams().
# A foreign key (e.g. supertrend_mult) now raises TypeError here, not silently no-ops.
STRATEGY_OVERRIDES = {}      # e.g. {"ema_fast": 12, "ema_slow": 26, "rsi_filter": False}
STRATEGY_CONFIG = dataclasses.replace(STRATEGY_CONFIG, **STRATEGY_OVERRIDES)

In [ ]:
# manual override — EXIT policy. None = each strategy's assigned default.
# Preset: EXIT_POLICY = EXIT_PRESETS["fixed_2pct_rr3"]()
# Custom: from engine.exits import CompositeExit, AtrStop, RrTarget
#         EXIT_POLICY = CompositeExit(AtrStop(1.5), RrTarget(2.0))
EXIT_POLICY = None

In [ ]:
# manual override — TRADE config (costs / sizing / leverage / direction). Empty = ACTIVE_TRADE.
# For direction: from engine.trade_configurator import TradeDirection
TRADE_OVERRIDES = {}         # e.g. {"leverage": 2.0, "direction": TradeDirection.LONG}
TRADING_CONFIG = dataclasses.replace(TRADING_CONFIG, **TRADE_OVERRIDES)

### Final configuration

In [4]:
# Prepare the final inputs the rest of the notebook uses.
# Runs after overrides.
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval

<a id="ema-rsi-section"></a>
## EMA + RSI

### Backtesting

In [5]:
# Import EMA + RSI strategy
from engine.strategies import EMACrossoverStrategy
STRATEGY = EMACrossoverStrategy

In [6]:
# Backtest EMA + RSI strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

════════════════════════════════════════════════════════════
  Backtest Summary: ema
  BTCUSDT | 15 | 800 bars
════════════════════════════════════════════════════════════
  Total trades      : 33
  Suppressed entries : 0  (blocked by direction/daily-loss gate)
  Win / Loss / BE    : 7 / 26 / 0
  Win rate           : 21.2%
  Total P&L (bps)    : -1287.0
  Avg P&L (bps)      : -39.0
  Max win (bps)      : +189.1
  Max loss (bps)     : -139.0
  Profit factor      : 0.35
  Max drawdown (bps) : 1629.2
  Sharpe (approx)    : -0.47
  ────────────────────────────────────────
  Initial equity     : 10,000.00
  Final equity       : 8,780.35
  Total return       : -12.20%
  Max drawdown       : 15.11%
  ────────────────────────────────────────
  Exits by reason:
    trailing_stop    : 24
    signal_flip      : 9
════════════════════════════════════════════════════════════


PosixPath('/Users/gm/Projects/tradekit/data/results/linear_BTCUSDT_15_last800/ema.json')

In [10]:
# Dollar P&L
# Per-trade dollar P&L — saved to <strategy>_trades.csv

trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])

net = result.final_equity - result.initial_equity
print(f"Final balance : ${result.final_equity:,.2f}")
print(f"Net profit    : ${net:+,.2f}")
print(f"Net return    : {result.total_return_pct:+.2f}%")
print(f"Max drawdown  : {result.max_drawdown_pct:.2f}%")
trades_pnl.head()

Final balance : $8,780.35
Net profit    : $-1,219.65
Net return    : -12.20%
Max drawdown  : 15.11%


,dir,pnl_bps,pnl_$,balance_after,exit_reason
0,long,-36.8,-36.80,9963.20,trailing_stop
1,short,185.6,184.88,10148.08,trailing_stop
2,long,-139.0,-141.06,10007.01,trailing_stop
3,short,56.3,56.31,10063.33,trailing_stop
4,long,-14.3,-14.42,10048.91,trailing_stop


In [7]:
# EMA + RSI strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

### Grid search

In [9]:
# Full grid search — Cartesian product across any of the four dimensions.
# Each grid is optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.
gs = grid_search(
    STRATEGY,
     strategy_grid={"ema_fast": [19, 21, 56], "ema_slow": [22, 30, 76]},
     trade_grid={"leverage": [1.0, 2.0]},
     exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
     data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

,interval,ema_fast,ema_slow,leverage,exit,trades,win_rate,total_pnl_bps,profit_factor,max_drawdown_bps,sharpe_approx,total_return_pct,final_equity
94,60,56,22,2.0,fixed_2pct_rr3,9,0.555556,360.638048,1.744889,424.000000,0.210455,6.838143,10683.814270
47,15,56,30,2.0,chandelier_2atr,9,0.555556,242.048870,2.150602,113.759490,0.230393,4.724558,10472.455823
45,15,56,30,2.0,default,9,0.555556,242.048870,2.150602,113.759490,0.230393,4.724558,10472.455823
100,60,56,30,2.0,fixed_2pct_rr3,8,0.500000,224.886817,1.498512,301.117322,0.160229,4.139777,10413.977688
91,60,56,22,1.0,fixed_2pct_rr3,9,0.555556,360.638048,1.744889,424.000000,0.210455,3.515931,10351.593075
42,15,56,30,1.0,default,9,0.555556,242.048870,2.150602,113.759490,0.230393,2.391701,10239.170077
44,15,56,30,1.0,chandelier_2atr,9,0.555556,242.048870,2.150602,113.759490,0.230393,2.391701,10239.170077
97,60,56,30,1.0,fixed_2pct_rr3,8,0.500000,224.886817,1.498512,301.117322,0.160229,2.161306,10216.130562
39,15,56,22,2.0,default,8,0.375000,74.065685,1.172714,248.119920,0.056382,1.120928,10112.092787
41,15,56,22,2.0,chandelier_2atr,8,0.375000,74.065685,1.172714,248.119920,0.056382,1.120928,10112.092787


In [11]:
# HEATMAP_METRIC is a configurable knob:
# flip between Sharpe / P&L / profit_factor / any grid_search column without editing the plot.
# Best value per ema_fast × ema_slow cell, across any other swept dimension.
# Renders only when the strategy grid is being swept.
HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"ema_fast", "ema_slow"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="ema_fast", columns="ema_slow", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="ema_slow", y="ema_fast", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs ema_fast × ema_slow swept in the grid above — nothing to plot.")

### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
# TRAIN_BARS is an in-sample window swept for the best params.
# TEST_BARS is an out-of-sample window the winner is then tested on.
# OBJECTIVE  can be any sweep metric: total_pnl_bps | sharpe_approx | profit_factor | ...
# MIN_TRADES lets ignore in-sample combos with fewer trades (noise, not signal)

GRID = {"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

════════════════════════════════════════════════════════════
  Walk-Forward (rolling) — 5 folds, objective=total_pnl_bps
  train=300 bars / test=100 bars
════════════════════════════════════════════════════════════
  OOS trades         : 7
  OOS total P&L (bps): -159.7
  OOS win rate       : 42.9%
  OOS profit factor  : 0.47
  OOS max DD (bps)   : 109.2
  OOS Sharpe (approx): -0.31
  ────────────────────────────────────────
  Σ in-sample  P&L (bps): -951.6
  Σ out-of-sample (bps) : -159.7
  WF efficiency (OOS/IS): n/a   (in-sample best was unprofitable — no edge to carry over)
════════════════════════════════════════════════════════════


In [13]:
# Per fold: the parameters chosen in-sample and their out-of-sample performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

,fold,test_start,test_end,ema_fast,ema_slow,is_total_pnl_bps,oos_total_pnl_bps,oos_trades,oos_win_rate
0,0,2026-06-06 05:15:00+00:00,2026-06-07 06:00:00+00:00,17,50,-121.942162,-133.487270,3,0.333333
1,1,2026-06-07 06:15:00+00:00,2026-06-08 07:00:00+00:00,17,50,-456.303442,0.000000,0,0.000000
2,2,2026-06-08 07:15:00+00:00,2026-06-09 08:00:00+00:00,17,50,-133.487270,-55.694035,2,0.500000
3,3,2026-06-09 08:15:00+00:00,2026-06-10 09:00:00+00:00,17,50,-104.520696,0.000000,0,0.000000
4,4,2026-06-10 09:15:00+00:00,2026-06-11 10:00:00+00:00,13,50,-135.366246,29.477414,2,0.500000


In [14]:
# Best parameters per fold.
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window
wf.param_stability()

,ema_fast,ema_slow
fold,,
0,17,50
1,17,50
2,17,50
3,17,50
4,13,50


In [15]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.
eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
px.line(eq, labels={"value": "equity", "index": ""},
        title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m").update_layout(showlegend=False).show()

In [16]:
# Walk-forward OOS trades (entries/exits), drawn with the EMAs each fold actually traded.
# Entries sit on real crosses: each fold's winning EMAs are recomputed and shown only over that fold's test window.
# The lines step at fold boundaries (the visible jump) is the re-optimisation.
# See wf.folds_frame() for the per-window params.

prepared_wf = df.copy()
prepared_wf["ema_fast"] = float("nan")
prepared_wf["ema_slow"] = float("nan")
for f in wf.folds:
    prep = STRATEGY(dataclasses.replace(STRATEGY_CONFIG, **f.best_params)).prepare(df)
    seg = (df.index >= f.test_start) & (df.index <= f.test_end)
    prepared_wf.loc[seg, ["ema_fast", "ema_slow"]] = prep.loc[seg, ["ema_fast", "ema_slow"]]

build_chart(prepared_wf, trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | per-fold EMAs").show()

### Monte Carlo simulations

In [17]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

════════════════════════════════════════════════════════════
  Monte Carlo — 10000 sims, block=5, 7 trades
════════════════════════════════════════════════════════════
  Terminal return %  : median -1.6   [p5 -3.5 … p95 +0.3]
  Max drawdown %     : median 2.2    [p5 1.1 … p95 3.7]
  P(profitable)      : 8.2%
════════════════════════════════════════════════════════════


In [18]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.
px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"},
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()
# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"},
             title="OOS max-drawdown distribution").show()

### Live signals

In [ ]:
# Live mode runs the same strategy / config / costs as the backtest above.
# It generates signals (tells you when to enter / exit). It does not place orders.
# The chart refreshes in the browser every poll_seconds.
# engine.run() blocks the execution of the rest of the notebook until stopped.

live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button

## Inverse EMA + RSI

Inverse EMA Crossover + RSI Filter \
Mean-reversion counterpart to EMA+RSI: fades the cross instead of riding it. \
Uses ATR(14) for dynamic trailing stops (adapts to volatility) and RSI(14) filter. \
Bets that EMA crosses mark momentum exhaustion, not continuation.

__How Inverse EMA+RSI Algorithm Determines Entry/Exit:__
- Fast EMA (9) / Slow EMA (21) — standard for crypto.
- Short Entry: Fast EMA crosses above Slow EMA AND RSI(14) > 30 (not oversold) — fading the bullish cross.
- Long Entry:  Fast EMA crosses below Slow EMA AND RSI(14) < 70 (not overbought) — fading the bearish cross.
- Exit: Opposite crossover (signal flip) OR price hits ATR-based trailing stop.
- Works best in range-bound / mean-reverting regimes; likely underperforms in strong trends.


<a id="inv-backtesting"></a>
### Backtesting

In [ ]:
# Import Inverse EMA + RSI strategy
from engine.strategies import InverseEMACrossoverStrategy
STRATEGY = InverseEMACrossoverStrategy

In [ ]:
# Backtest Inverse EMA + RSI strategy
strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=TRADING_CONFIG)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, DATA_CONFIG)

In [ ]:
# Per-trade dollar P&L — the engine's real figures (TradingConfig sizing +
# initial_equity), already on each Trade and saved to <strategy>_trades.csv.

trades_pnl = pd.DataFrame([{
    "dir": t.direction.value,
    "pnl_bps": round(t.pnl_bps, 1),
    "pnl_$": round(t.pnl_currency, 2),
    "balance_after": round(t.equity_after, 2),
    "exit_reason": t.exit_reason.value if t.exit_reason else None,
} for t in result.trades])

net = result.final_equity - result.initial_equity
print(f"Final balance : ${result.final_equity:,.2f}")
print(f"Net profit    : ${net:+,.2f}")
print(f"Net return    : {result.total_return_pct:+.2f}%")
print(f"Max drawdown  : {result.max_drawdown_pct:.2f}%")
trades_pnl

In [ ]:
# Inverse EMA + RSI strategy chart — exits coloured by reason (P&L on hover) + entry→exit paths.
prepared = strategy.prepare(df)
build_chart(
    prepared, trades=result.trades,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

<a id="inv-grid-search"></a>
### Grid search

In [ ]:
# Full grid search — Cartesian product across any of the four dimensions. Each grid is
# optional: uncomment the ones you want to sweep, leave the rest commented to hold them fixed.
# Row count = |strategy| × |trade| × |exit| × |data|, so keep grids tight.
# Rank by total_return_pct: pnl_bps is sizing-invariant, so trade-knob sweeps only move equity.
gs = grid_search(
    STRATEGY,
    # strategy_grid={"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]},
    # trade_grid={"leverage": [1.0, 2.0]},
    # exit_grid=[None, "fixed_2pct_rr3", "chandelier_2atr"],
    # data_grid={"interval": ["15", "60"]},   # reloads per spec
    base_config=STRATEGY_CONFIG, base_trading=TRADING_CONFIG, base_data=DATA_CONFIG,
)
gs.sort_values("total_return_pct", ascending=False).head(12)

In [ ]:
# HEATMAP_METRIC is a configurable knob: flip between Sharpe / P&L / profit_factor / any
# grid_search column without editing the plot. (Best value per ema_fast × ema_slow cell,
# across any other swept dimension.) Rendered only when the strategy grid was swept.
HEATMAP_METRIC = "total_pnl_bps"   # any grid_search column
if {"ema_fast", "ema_slow"}.issubset(gs.columns):
    # Diverging colour split at the metric's breakeven: P&L/Sharpe at 0, profit_factor at 1, win_rate at 0.5.
    midpoint = {"profit_factor": 1.0, "win_rate": 0.5}.get(HEATMAP_METRIC, 0.0)
    px.imshow(
        gs.pivot_table(index="ema_fast", columns="ema_slow", values=HEATMAP_METRIC, aggfunc="max"),
        color_continuous_scale="RdYlGn", color_continuous_midpoint=midpoint, aspect="auto",
        labels=dict(x="ema_slow", y="ema_fast", color=HEATMAP_METRIC),
        title=f"In-sample {HEATMAP_METRIC} — {strategy.name} | {SYMBOL} {INTERVAL}m",
    ).show()
else:
    print("Heatmap needs ema_fast × ema_slow swept in the grid above — nothing to plot.")

<a id="inv-walk-forward"></a>
### Walk-forward analysis

In [ ]:
# Walk-forward re-sweeps the strategy grid every train window.
GRID = {"ema_fast": [5, 9, 13, 17], "ema_slow": [20, 30, 40, 50]}
MIN_TRADES = 2
TRAIN_BARS, TEST_BARS = 300, 100
OBJECTIVE = "total_pnl_bps"
wf = walk_forward(
    STRATEGY, df, GRID,
    train_bars=TRAIN_BARS, test_bars=TEST_BARS,
    symbol=SYMBOL, interval=INTERVAL, trading_config=TRADING_CONFIG, exit_policy=EXIT_POLICY,
    base_config=STRATEGY_CONFIG,   # non-swept knobs come from STRATEGY_CONFIG (automatic + manual)
    objective=OBJECTIVE, min_trades=MIN_TRADES,
)
print(wf.summary())

In [ ]:
# Per fold: the parameters chosen in-sample and their out-of-sample performance.
# Parameters that vary significantly across folds may indicate an unstable optimization and overfitting.
wf.folds_frame()

In [ ]:
# Best parameters per fold.
# Stable across folds = trustworthy; jumpy = tend to overfit, unlikely to work next window
wf.param_stability()

In [ ]:
# Out-of-sample equity curve: the stitched test windows compounded.
# The equity path you'd have lived through re-tuning periodically on only past data.
eq = wf.equity_curve(initial=TRADING_CONFIG.initial_equity)
px.line(eq, labels={"value": "equity", "index": ""},
        title=f"Walk-forward OOS equity — {strategy.name} | {SYMBOL} {INTERVAL}m").update_layout(showlegend=False).show()

In [ ]:
# Walk-forward OOS trades (entries/exits), drawn with the EMAs each fold actually traded.
# Entries sit on real crosses: each fold's winning EMAs are recomputed and shown only over that fold's test window.
# The lines step at fold boundaries (the visible jump) is the re-optimisation.
# See wf.folds_frame() for the per-window params.

prepared_wf = df.copy()
prepared_wf["ema_fast"] = float("nan")
prepared_wf["ema_slow"] = float("nan")
for f in wf.folds:
    prep = STRATEGY(dataclasses.replace(STRATEGY_CONFIG, **f.best_params)).prepare(df)
    seg = (df.index >= f.test_start) & (df.index <= f.test_end)
    prepared_wf.loc[seg, ["ema_fast", "ema_slow"]] = prep.loc[seg, ["ema_fast", "ema_slow"]]

build_chart(prepared_wf, trades=wf.oos_trades,
            title=f"Walk-forward OOS trades — {strategy.name} | per-fold EMAs").show()

<a id="inv-monte-carlo"></a>
### Monte Carlo simulations

In [ ]:
# The spread shows the range of results that could occur due to luck, not just the outcome that actually happened.
# It answers: how much did trade ordering affect the result, and how severe could the drawdown realistically be?
# P(profitable) near 50% means the OOS edge is equivalent to noise (close to a random chance).

mc = monte_carlo(wf.oos_trades, n_sims=10_000, block=5, seed=0,
                 initial_equity=TRADING_CONFIG.initial_equity)
print(mc.summary())

In [ ]:
# How those thousands of shuffled runs ended up.
# Wide / leaning negative = the result leans on luck; tight / mostly positive = sturdier edge.
px.histogram(x=mc.samples["terminal_return_pct"], nbins=60,
             labels={"x": "terminal return %"},
             title=f"OOS terminal-return distribution — {strategy.name} | {mc.n_sims} sims").show()
# Worst peak-to-trough drop in each run — how deep a drawdown to be ready for.
px.histogram(x=mc.samples["max_drawdown_pct"], nbins=60,
             labels={"x": "max drawdown %"},
             title="OOS max-drawdown distribution").show()

<a id="inv-live-signals"></a>
### Live signals

In [ ]:
# Live mode runs the same strategy / config / costs as the backtest above.
# It generates signals (tells you when to enter / exit). It does not place orders.
# The chart refreshes in the browser every poll_seconds.
# engine.run() blocks the execution of the rest of the notebook until stopped.

live_strategy = STRATEGY(STRATEGY_CONFIG, exit_policy=EXIT_POLICY)
engine = LiveEngine(
    strategy=live_strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=10,
    trading_config=TRADING_CONFIG,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{live_strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{live_strategy.name}.db"),
)
engine.run()    # to stop: interrupt the kernel/ use the Stop button